<a href="https://colab.research.google.com/github/laugarcias/FIAP--3/blob/main/aula_spark_hadson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# Instalando pacote PySpark
!pip install pyspark

In [8]:
#Instalação de funções básicas para o Spark
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.functions import countDistinct

Iniciando Ambiente Spark e Criando a Sessão SparkSession

In [9]:
spark = (
    SparkSession
    .builder
    .master("local[*]")
    .getOrCreate()


)

In [10]:
print(spark.version)

4.0.4


Importação de Bases.csv

In [11]:
df_clientes = spark.read.csv('clientes.csv', sep = ";", inferSchema= True, header = True)
df_vendas = spark.read.csv('vendas.csv', sep = ",", inferSchema= True, header = True)

In [12]:
df_clientes.show()
df_vendas.show()

+----------+--------------------+--------------------+-----+--------------------+---+---------+-------------+
|cliente_id|                nome|               email|idade|              cidade| UF|     sexo|data_cadastro|
+----------+--------------------+--------------------+-----+--------------------+---+---------+-------------+
|         1|        Miguel Porto|eloah69@nasciment...|   51|Sao Bernardo do C...| SP| Feminino|   24/09/2020|
|         2|Gustavo Henrique ...|   ubarros@gmail.com|   62|            Londrina| PR| Feminino|   28/05/2024|
|         3|       Julia Correia|ana-lauramoraes@u...|   65|            Sao Luis| MA| Feminino|   08/01/2020|
|         4|        Pietra Sales| usilveira@ig.com.br|   45|              Recife| PE|Masculino|   22/05/2021|
|         5|    Guilherme Barros|davi-luizfarias@b...|   24|              Cuiaba| MT| Feminino|   20/04/2021|
|         6|Sr. Guilherme Rez...|cavalcantieloah@b...|   57|Sao Bernardo do C...| SP|Masculino|   13/08/2023|
|         

Exemplo de RDD

In [13]:
dados_clientes = [
    (1,"Laura Garcias", "barretolaura775@gmail.com", 24,"São Pualo", "SP"),
    (2,"Marcos Viniciues da mata Ribeiro", "damatamarcos@gmail.com",28,"Curitiba","PR")
]

Não é utilizado mais RDD - esta "obsoleto" somente para momentos específicos

In [14]:
rdd_clientes = spark.sparkContext.parallelize(dados_clientes)

In [15]:
rdd_clientes.collect()

[(1, 'Laura Garcias', 'barretolaura775@gmail.com', 24, 'São Pualo', 'SP'),
 (2,
  'Marcos Viniciues da mata Ribeiro',
  'damatamarcos@gmail.com',
  28,
  'Curitiba',
  'PR')]

Exploração Inicial das Bases Importadas ( Clientes e Vendas)

In [16]:
df_clientes.printSchema()

root
 |-- cliente_id: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- email: string (nullable = true)
 |-- idade: integer (nullable = true)
 |-- cidade: string (nullable = true)
 |-- UF: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- data_cadastro: string (nullable = true)



In [17]:
df_vendas.printSchema()

root
 |-- venda_id: integer (nullable = true)
 |-- data_venda: date (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- produto: string (nullable = true)
 |-- quantidade: integer (nullable = true)
 |-- valor_unitario: double (nullable = true)



In [18]:
df_clientes.show(5)
df_vendas.show(5)

+----------+--------------------+--------------------+-----+--------------------+---+---------+-------------+
|cliente_id|                nome|               email|idade|              cidade| UF|     sexo|data_cadastro|
+----------+--------------------+--------------------+-----+--------------------+---+---------+-------------+
|         1|        Miguel Porto|eloah69@nasciment...|   51|Sao Bernardo do C...| SP| Feminino|   24/09/2020|
|         2|Gustavo Henrique ...|   ubarros@gmail.com|   62|            Londrina| PR| Feminino|   28/05/2024|
|         3|       Julia Correia|ana-lauramoraes@u...|   65|            Sao Luis| MA| Feminino|   08/01/2020|
|         4|        Pietra Sales| usilveira@ig.com.br|   45|              Recife| PE|Masculino|   22/05/2021|
|         5|    Guilherme Barros|davi-luizfarias@b...|   24|              Cuiaba| MT| Feminino|   20/04/2021|
+----------+--------------------+--------------------+-----+--------------------+---+---------+-------------+
only showi

Seleção de Colunas

In [19]:
df_clientes.select('nome','cidade').show(5)

+--------------------+--------------------+
|                nome|              cidade|
+--------------------+--------------------+
|        Miguel Porto|Sao Bernardo do C...|
|Gustavo Henrique ...|            Londrina|
|       Julia Correia|            Sao Luis|
|        Pietra Sales|              Recife|
|    Guilherme Barros|              Cuiaba|
+--------------------+--------------------+
only showing top 5 rows


In [20]:
df_clientes.select('cidade')\
           .distinct()\
           .orderBy('cidade', ascending = True)\
           .show()

+------------------+
|            cidade|
+------------------+
|           Aracaju|
|Balneario Camboriu|
|             Belem|
|    Belo Horizonte|
|          Blumenau|
|          Brasilia|
|          Campinas|
|      Campo Grande|
|            Cuiaba|
|          Curitiba|
|     Florianopolis|
|         Fortaleza|
|           Goiania|
|         Guarulhos|
|       Joao Pessoa|
|         Joinville|
|           Jundiai|
|          Londrina|
|            Maceio|
|            Manaus|
+------------------+
only showing top 20 rows


In [21]:
df_clientes.select(countDistinct('cidade')\
                   .alias("qtd.cidades"))\
                   .show()


+-----------+
|qtd.cidades|
+-----------+
|         39|
+-----------+



Seleção de Colunas Definindo parametro do Conjunto de colunas

In [22]:
col_1 = list(set(df_clientes.columns)-{'email', 'idade', 'cliente_id'})

df_clientes2 = df_clientes\
   .select(*col_1)\
   .distinct()\
   .orderBy('cidade', ascending = True)\
   .show()

+---------+---+-------+-------------+--------------------+
|     sexo| UF| cidade|data_cadastro|                nome|
+---------+---+-------+-------------+--------------------+
| Feminino| SE|Aracaju|   28/02/2023|  Dra. Yasmin Taboao|
| Feminino| SE|Aracaju|   17/06/2022|     Rafaela Peixoto|
|Masculino| SE|Aracaju|   17/12/2024|      Melissa Farias|
|Masculino| SE|Aracaju|   20/12/2021|       Vitor da Cruz|
|Masculino| SE|Aracaju|   24/09/2020|    Vitória Carvalho|
|Masculino| SE|Aracaju|   25/10/2023|Marcos Vinicius B...|
| Feminino| SE|Aracaju|   27/04/2022|Sr. Guilherme Rez...|
| Feminino| SE|Aracaju|   04/11/2023|    Joaquim Ferreira|
| Feminino| SE|Aracaju|   29/01/2023|        Rafaela Melo|
| Feminino| SE|Aracaju|   18/08/2024| Luiz Felipe Almeida|
| Feminino| SE|Aracaju|   24/09/2020|        Paulo Mendes|
| Feminino| SE|Aracaju|   22/04/2024|       Vitor da Cruz|
|Masculino| SE|Aracaju|   09/10/2020|    Heloísa Oliveira|
| Feminino| SE|Aracaju|   24/11/2022|     Dra. Nina Cunh

Condição de Filtro com Where

In [23]:
df_clientes.where("idade>=30" ).show()

+----------+--------------------+--------------------+-----+--------------------+---+---------+-------------+
|cliente_id|                nome|               email|idade|              cidade| UF|     sexo|data_cadastro|
+----------+--------------------+--------------------+-----+--------------------+---+---------+-------------+
|         1|        Miguel Porto|eloah69@nasciment...|   51|Sao Bernardo do C...| SP| Feminino|   24/09/2020|
|         2|Gustavo Henrique ...|   ubarros@gmail.com|   62|            Londrina| PR| Feminino|   28/05/2024|
|         3|       Julia Correia|ana-lauramoraes@u...|   65|            Sao Luis| MA| Feminino|   08/01/2020|
|         4|        Pietra Sales| usilveira@ig.com.br|   45|              Recife| PE|Masculino|   22/05/2021|
|         6|Sr. Guilherme Rez...|cavalcantieloah@b...|   57|Sao Bernardo do C...| SP|Masculino|   13/08/2023|
|         8|Luiz Fernando Car...|joao-guilhermemor...|   55|       Florianopolis| SC| Feminino|   29/04/2021|
|         

In [24]:
df_clientes.where("idade between 0 and 18" ).show()

+----------+--------------------+--------------+-----+--------------+---+---------+-------------+
|cliente_id|                nome|         email|idade|        cidade| UF|     sexo|data_cadastro|
+----------+--------------------+--------------+-----+--------------+---+---------+-------------+
|       117|Sr. Emanuel da Co...|renan57@da.net|   18|      Sao Luis| MA|Masculino|   10/02/2020|
|       251|Sr. Emanuel da Co...|renan57@da.net|   18|Rio de Janeiro| RJ|Masculino|   14/09/2022|
|       276|Sr. Emanuel da Co...|renan57@da.net|   18|        Cuiaba| MT| Feminino|   10/09/2023|
|       794|Sr. Emanuel da Co...|renan57@da.net|   18|     Fortaleza| CE|Masculino|   30/06/2021|
+----------+--------------------+--------------+-----+--------------+---+---------+-------------+



Removendo Registros Duplicados

In [25]:
df_clientes.count()

1000

In [26]:
df_vendas.count()

1500

In [27]:
df_vendas = df_vendas.dropDuplicates()
df_clientes = df_clientes.dropDuplicates()


Applied Analytcs - Responda os questionamentos

1 - Qual cidade gerou o maior valor total de venda?
2 - Qual produto mais vendido em quantidade e qual a sua média unitária?

In [28]:
# Pergunta 01

df_join = df_vendas.join( df_clientes, on ="cliente_id", how = 'inner')
df_join.show(5)

+----------+--------+----------+-------+----------+--------------+-------------------+--------------------+-----+---------+---+---------+-------------+
|cliente_id|venda_id|data_venda|produto|quantidade|valor_unitario|               nome|               email|idade|   cidade| UF|     sexo|data_cadastro|
+----------+--------+----------+-------+----------+--------------+-------------------+--------------------+-----+---------+---+---------+-------------+
|        50|     785|2025-01-19| Tablet|         2|        580.64|    Bernardo Farias|       zpires@da.com|   49|  Jundiai| SP|Masculino|   10/10/2021|
|       154|     551|2025-01-04| Tablet|         4|       3969.83|      Julia Correia|ana-lauramoraes@u...|   65|   Recife| PE|Masculino|   09/07/2020|
|       154|     454|2025-01-29|Teclado|         4|        265.91|      Julia Correia|ana-lauramoraes@u...|   65|   Recife| PE|Masculino|   09/07/2020|
|       313|    1020|2025-02-20|Teclado|         1|       1757.54|João Vitor Monteiro|da

In [29]:
cidade_maior_venda = df_join\
      .groupBy('cidade')\
      .agg(F.round(F.sum(F.col('quantidade')* F.col('valor_unitario')),2).alias('total_venda'))\
      .orderBy('total_venda', ascending = False)\
      .show(1)


+-------+-----------+
| cidade|total_venda|
+-------+-----------+
|Niteroi|  502446.27|
+-------+-----------+
only showing top 1 row


In [31]:
#Pergunta 02
produto_mais_vendido = df_vendas\
      .groupBy('produto')\
      .agg(F.round(F.sum(F.col('quantidade')),2).alias('total_quantidade'),\
           F.round(F.avg('valor_unitario'),2).alias('media_unitaria'))\
      .orderBy('total_quantidade', ascending = False)\
      .show(5)


+----------+----------------+--------------+
|   produto|total_quantidade|media_unitaria|
+----------+----------------+--------------+
|    Tablet|             886|       3100.82|
|Impressora|             685|       2864.13|
|  Notebook|             633|       2648.59|
|   Teclado|             549|       1875.65|
|   Celular|             449|       2719.12|
+----------+----------------+--------------+
only showing top 5 rows


Testando Consultas via Spark

In [33]:
#Criação de view para utilização no sparkSQL
df_clientes.createOrReplaceTempView('clientes')
df_vendas.createOrReplaceTempView('vendas')

In [39]:
spark.sql(

'''
 SELECT  COUNT(DISTINCT CIDADE) AS QTD_CIDADES
 FROM CLIENTES

'''
).show(5)

+-----------+
|QTD_CIDADES|
+-----------+
|         39|
+-----------+



In [44]:
spark.sql(

'''
 SELECT DISTINCT CIDADE
 FROM CLIENTES
 ORDER BY 1


'''
).show(5)

+------------------+
|            CIDADE|
+------------------+
|           Aracaju|
|Balneario Camboriu|
|             Belem|
|    Belo Horizonte|
|          Blumenau|
+------------------+
only showing top 5 rows


In [46]:
spark.sql(

'''
 SELECT CIDADE, COUNT (DISTINCT CLIENTE_ID ) AS QTD_CLIENTES
 FROM CLIENTES
 GROUP BY CIDADE
 ORDER BY QTD_CLIENTES DESC


'''
).show(5)

+-------------------+------------+
|             CIDADE|QTD_CLIENTES|
+-------------------+------------+
|             Santos|          40|
|            Niteroi|          38|
|           Sorocaba|          33|
|Sao Jose dos Campos|          33|
|           Salvador|          31|
+-------------------+------------+
only showing top 5 rows


In [55]:
#Pergunta 03 - Top 5 clientes com mais compras de produtos

spark.sql(

'''
 SELECT DISTINCT
 A.NOME AS NOME_CLIENTE,
 SUM(B.QUANTIDADE) AS QTD_COMPRAS
 FROM CLIENTES AS A
 LEFT JOIN VENDAS AS B ON (A.CLIENTE_ID = B.CLIENTE_ID)
 GROUP BY A.NOME
 ORDER BY QTD_COMPRAS DESC
 LIMIT 5


'''
).show()

+-------------------+-----------+
|       NOME_CLIENTE|QTD_COMPRAS|
+-------------------+-----------+
|       Calebe Nunes|        128|
|    Bernardo da Luz|         98|
|     Eduardo Campos|         98|
|João Vitor Monteiro|         88|
|         Luna Gomes|         85|
+-------------------+-----------+



In [56]:
#Qual foi a data com maior total de vendas

spark.sql(

'''
 SELECT DATA_VENDA,
  ROUND(SUM(QUANTIDADE * VALOR_UNITARIO)) AS TOTAL_VENDAS
 FROM VENDAS
 GROUP BY DATA_VENDA
 ORDER BY TOTAL_VENDAS DESC
 LIMIT 1


'''
).show()

+----------+------------+
|DATA_VENDA|TOTAL_VENDAS|
+----------+------------+
|2025-01-08|    661362.0|
+----------+------------+



In [58]:
df_final = (

    df_join
    .withColumnRenamed('quantidade', 'qtd_vendas')
    .withColumnRenamed('produto', 'nome_produto')
            ).show(5)

+----------+--------+----------+------------+----------+--------------+-------------------+--------------------+-----+---------+---+---------+-------------+
|cliente_id|venda_id|data_venda|nome_produto|qtd_vendas|valor_unitario|               nome|               email|idade|   cidade| UF|     sexo|data_cadastro|
+----------+--------+----------+------------+----------+--------------+-------------------+--------------------+-----+---------+---+---------+-------------+
|        50|     785|2025-01-19|      Tablet|         2|        580.64|    Bernardo Farias|       zpires@da.com|   49|  Jundiai| SP|Masculino|   10/10/2021|
|       154|     551|2025-01-04|      Tablet|         4|       3969.83|      Julia Correia|ana-lauramoraes@u...|   65|   Recife| PE|Masculino|   09/07/2020|
|       154|     454|2025-01-29|     Teclado|         4|        265.91|      Julia Correia|ana-lauramoraes@u...|   65|   Recife| PE|Masculino|   09/07/2020|
|       313|    1020|2025-02-20|     Teclado|         1|  